# Longest Increasing Subsequence (LIS) and Path Reconstruction

**Session:** 2026-11-08 · Dynamic Programming and Divide-and-Conquer  
**Module:** 01 · Python Fundamentals and Algorithmic Thinking  

---

## 1. Problem Formulation

Given a sequence of integers $S = [a_0, a_1, \dots, a_{n-1}]$, the **Longest Increasing Subsequence (LIS)** problem seeks the maximal length of a subsequence $a_{i_0}, a_{i_1}, \dots, a_{i_{k-1}}$ such that:
$$i_0 < i_1 < \dots < i_{k-1} \quad \text{and} \quad a_{i_0} < a_{i_1} < \dots < a_{i_{k-1}}$$

Notice that a **subsequence** does *not* require contiguous elements, unlike a **substring** or **subarray**.


In [ ]:
from __future__ import annotations

import bisect
import json
from pathlib import Path

DATA_DIR = Path("../data")
with (DATA_DIR / "lis_sequences.json").open(encoding="utf-8") as f:
    lis_cases = json.load(f)

## 2. Dynamic Programming Formulation $O(n^2)$

### State Definition:
Let $dp[i]$ denote the length of the longest strictly increasing subsequence whose **terminal element** is $nums[i]$.

### Base Case:
Every single element forms an increasing subsequence of length 1:
$$dp[i] = 1 \quad \forall \, i \in [0, n-1]$$

### State Transition Recurrence:
For each index $i$ from $0$ to $n-1$, and for each prior index $j < i$:
$$dp[i] = \max_{0 \le j < i, \, nums[j] < nums[i]} (dp[j] + 1)$$

The overall maximum LIS length is:
$$\text{LIS}(nums) = \max_{0 \le i < n} dp[i]$$


In [ ]:
def longest_subsequence(nums: list[int]) -> tuple[int, list[int]]:
    """Basic LIS length calculation matching lecture formulation."""
    n = len(nums)
    if n == 0:
        return 0, []

    dp = [1] * n
    for i in range(n):
        for j in range(i):
            if nums[i] > nums[j]:
                dp[i] = max(dp[i], dp[j] + 1)

    return max(dp), dp


# Verify on the lecture example: nums = [3, 10, 2, 1, 20]
lecture_nums = lis_cases["lecture_example"]["nums"]
max_len, dp_table = longest_subsequence(lecture_nums)
print("Input nums:", lecture_nums)
print("DP table:  ", dp_table)
print("Max LIS:   ", max_len)
assert max_len == lis_cases["lecture_example"]["expected_length"]

## 3. Backtracking Path Reconstruction (`trace_dp`)

To reconstruct the actual elements of the subsequence, we maintain a `trace_dp` array where `trace_dp[i]` stores the predecessor index $j$ that yielded the optimal $dp[i]$. 

We trace backwards from the index of maximum $dp$, and then reverse the recovered list.


In [ ]:
def longest_subsequence_with_trace(
    nums: list[int],
) -> tuple[int, list[int], list[int]]:
    """LIS with predecessor pointer tracing to reconstruct optimal sequence."""
    n = len(nums)
    if n == 0:
        return 0, [], []

    dp = [1] * n
    trace_dp = [-1] * n

    for i in range(n):
        for j in range(i):
            if nums[i] > nums[j] and dp[i] < dp[j] + 1:
                dp[i] = dp[j] + 1
                trace_dp[i] = j

    max_len = max(dp)
    idx_max = dp.index(max_len)

    result = []
    curr = idx_max
    while curr != -1:
        result.append(nums[curr])
        curr = trace_dp[curr]

    result.reverse()  # Reconstruct in forward sequence order
    return max_len, dp, result


max_len, dp_table, path = longest_subsequence_with_trace(lecture_nums)
print("Reconstructed Subsequence:", path)
print("Predecessor trace_dp:     ", dp_table)
assert path == lis_cases["lecture_example"]["example_subsequence"]

## 4. Step-by-Step State Transition Trace

Let us display the step-by-step progress table for `nums = [3, 10, 2, 1, 20]` as analyzed in the lecture slides.


In [ ]:
print(f"{'Index (i)':<10} | {'Value nums[i]':<14} | {'dp[i]':<8} | {'Active Subsequence'}")
print("-" * 65)
for i in range(len(lecture_nums)):
    _, _, p = longest_subsequence_with_trace(lecture_nums[: i + 1])
    print(f"{i:<10} | {lecture_nums[i]:<14} | {dp_table[i]:<8} | {p}")

## 5. Optimal $O(n \log n)$ Patience Sorting Algorithm

While $O(n^2)$ DP is foundational, LIS can be solved in $O(n \log n)$ time using **Patience Sorting** with binary search. 

We maintain an array `tails`, where `tails[k]` stores the smallest tail of all increasing subsequences of length $k+1$ found so far.


In [ ]:
def longest_subsequence_fast(nums: list[int]) -> int:
    """Patience sorting with binary search O(n log n)."""
    tails: list[int] = []
    for x in nums:
        idx = bisect.bisect_left(tails, x)
        if idx == len(tails):
            tails.append(x)
        else:
            tails[idx] = x
    return len(tails)


assert longest_subsequence_fast(lecture_nums) == 3
print("Fast O(n log n) Patience Sorting verified.")

## 6. Automated Validation Across All Benchmark Sequences


In [ ]:
for name, case in lis_cases.items():
    nums = case["nums"]
    expected_len = case["expected_length"]
    computed_len, _, path = longest_subsequence_with_trace(nums)
    fast_len = longest_subsequence_fast(nums)

    assert computed_len == expected_len, (
        f"Failed {name}: got {computed_len}, expected {expected_len}"
    )
    assert fast_len == expected_len, f"Fast LIS mismatch on {name}"
    print(f"PASS: {name:<20} | Length: {computed_len} | Candidate: {path}")